# Trinity Reservoir Scraper

### This script downloads the daily and monthly Trinity reservoir data using the scrapper built by WAPA

In [ ]:
#All kept from earlier code (Some can be removed)
import requests
import sys
import csv
import json
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import io
import seaborn as sns
import numpy as np
from copy import copy
from scraper import Scraper

In [ ]:
#Select the start date (runs till current time)
start_string = '1995-10-01'
dsf = "%Y-%m-%d"

In [ ]:
# labels_basin - data types that are common to all basins/reservoirs along with the reservoir key
# station_basin - reservoir key as used in CALFEWS
labels_basin = ['snow', 'fnf', 'storage', 'inf', 'otf', 'evap', 'precip', 'gains', 'fci']
stations_basin = ['TRT']


In [ ]:
# this initializes the object that will become the input scenario
calfews_data = Scraper(stations_basin, labels_basin, timestep = 'd', start_date = start_string, date_string_format = dsf)


In [ ]:
#Checks
calfews_data.data_timeseries

In [ ]:
# monthly inputs (extend further back) Speak to Dan about QA/QC
start_string_monthly = '1905-10-01'
labels_basin_monthly = ['fnf', 'inf']
calfews_data_m = Scraper(stations_basin, labels_basin_monthly, timestep = 'm', start_date = start_string_monthly, date_string_format = dsf)


In [ ]:
#setup input scenario data file with all key/data type pairs (Check)
calfews_data.initialize_dataframe()
calfews_data_m.initialize_dataframe()

In [ ]:
##Monthly Stations Column Lists:
# when CDEC data uses different keys than the main 'basin key'
# we need to associate that key with the main 'basin key' for a specific data type
calfews_data_m.add_station_map(stations_basin, stations_basin, ['inf',])

#Changing Trinity Monthly Inflows to Clair Engle Lake
calfews_data_m.add_station_map(['TRT',], ['CLE',], ['inf',])

In [ ]:
# these are the station keys for monthly full-natural flow associated with each key in stations_basin
# TNL - Trinity River at Lewiston has FNF records going back to 1911
stations_monthly_fnf = ['TNL']
calfews_data_m.add_station_map(stations_basin, stations_monthly_fnf, ['fnf',])



In [ ]:
#Checks
calfews_data_m.station_use

In [ ]:
# these are the station keys for daily downstream incremental flows associated with each key in stations_basin
#None added for Trinity in second position
station_names = ['none']
station_types = ['gains',]
calfews_data.add_station_map(stations_basin, station_names, station_types)

In [ ]:
#Checks
calfews_data.station_use

In [ ]:
# different station keys for Trinity
calfews_data.add_station_map(['TRT',], ['TNL'], ['fnf',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['storage',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['inf',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['otf',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['evap',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['precip',])
calfews_data.add_station_map(['TRT',], ['none'], ['fci',]) #--- This is hard coded in the model where the fcr is computed

In [ ]:
#Checks
calfews_data.station_use

In [ ]:
# snow station mapping (Trinity Values are added)
basin_names = ['TRT']
station_names = [['MUM', 'BNK', 'RRM']]
calfews_data.add_station_map(basin_names, station_names, ['snow',])

In [ ]:
##basin-level data
sensor_basin = [82, 8, 15, 76, 23, 74, 45, 41]
labels_basin = ['snow', 'fnf', 'storage', 'inf', 'otf', 'evap', 'precip', 'gains']
url_parts_d = ['https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet?Stations=', '&SensorNums=', '&dur_code=D&Start=', '&End=']
url_parts_m = ['https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet?Stations=', '&SensorNums=', '&dur_code=M&Start=', '&End=']
url_parts_h = ['https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet?Stations=', '&SensorNums=', '&dur_code=H&Start=', '&End=']


In [ ]:
hourly_dict = {}
for x in stations_basin:
  hourly_dict[x] = []

In [ ]:
#Checks
calfews_data.station_use

In [ ]:
# link and read all data
calfews_data_m.link_api(stations_basin, [76,], ['inf',], url_parts_m, url_parts_h, hourly_dict)
calfews_data_m.link_api(stations_basin, [65,], ['fnf',], url_parts_m, url_parts_h, hourly_dict)
calfews_data.link_api(stations_basin, sensor_basin, labels_basin, url_parts_d, url_parts_h, hourly_dict)
calfews_data_m.find_ratios(stations_basin, 'inf')
calfews_data.fill_missing(stations_basin, calfews_data_m.station_coefs, calfews_data_m.real_time_data)
calfews_data.adjust_fnf_monthly(calfews_data_m.real_time_data, stations_basin)


In [ ]:
#Save both files as needed
calfews_data.real_time_data
#calfews_data_m.real_time_data